In [1]:
import os
import sys
import json
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download, HfApi

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

print("Working directory:", os.getcwd())
print("Python:", sys.version)

g:\work\MLops\mlops-bootcamp\hw22_deployment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: g:\work\MLops\mlops-bootcamp\hw22_deployment\HW3_A
Python: 3.11.7 | packaged by Anaconda, Inc. | (main, Dec 15 2023, 18:05:47) [MSC v.1916 64 bit (AMD64)]


# Step 1: Download model from HuggingFace

Download `sentence-transformers/all-MiniLM-L6-v2` and save 6 files to `bundle/model/`:

| # | File |
|---|------|
| 1 | `config.json` |
| 2 | `tokenizer_config.json` |
| 3 | `tokenizer.json` |
| 4 | `vocab.txt` |
| 5 | `special_tokens_map.json` |
| 6 | `model.safetensors` |

Use `snapshot_download` from `huggingface_hub` or download manually with `AutoModel.save_pretrained()` + `AutoTokenizer.save_pretrained()`.

After downloading, save the git commit hash to `bundle/model/.commit`.

In [2]:
# Download the 6 model files to bundle/model/

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
BUNDLE_MODEL_DIR = Path("bundle/model")
BUNDLE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.txt",
    "special_tokens_map.json",
    "model.safetensors",
]

snapshot_download(
    repo_id=MODEL_ID,
    revision="main",
    local_dir=str(BUNDLE_MODEL_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=required_files,
)

commit = HfApi().model_info(MODEL_ID, revision="main").sha
(BUNDLE_MODEL_DIR / ".commit").write_text(commit + "\n", encoding="utf-8")

print("Downloaded model files to:", BUNDLE_MODEL_DIR)
print("Model commit hash:", commit)

# Verify all 6 files exist
for fname in required_files:
    fpath = BUNDLE_MODEL_DIR / fname
    assert fpath.exists(), f"MISSING: {fname}"
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  [OK] {fname:35s} {size_mb:7.1f} MB")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]g:\work\MLops\mlops-bootcamp\hw22_deployment\.venv\Lib\site-packages\huggingface_hub\file_download.py:1212: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 122.45it/s]


Downloaded model files to: bundle\model
Model commit hash: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
  [OK] config.json                             0.0 MB
  [OK] tokenizer_config.json                   0.0 MB
  [OK] tokenizer.json                          0.4 MB
  [OK] vocab.txt                               0.2 MB
  [OK] special_tokens_map.json                 0.0 MB
  [OK] model.safetensors                      86.7 MB


# Step 2: Write metadata.json

Create `bundle/metadata.json` with accurate information about the bundle.
Required fields:
- `model_name`: the HuggingFace model ID
- `model_revision`: the git commit hash from the `.commit` file
- `embedding_dim`: 384
- `max_seq_len`: 256
- `framework_version`: the installed torch version
- `transformers_version`: the installed transformers version
- `built_by`: YOUR NAME
- `build_timestamp_utc`: current time in ISO 8601 UTC format

In [3]:
# Create bundle/metadata.json
import importlib


commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
torch_version = torch.__version__
transformers_version = importlib.metadata.version("transformers")

metadata = {
    "model_name": MODEL_ID,
    "model_revision": commit,
    "embedding_dim": 384,
    "max_seq_len": 256,
    "framework_version": torch_version,
    "transformers_version": transformers_version,
    "built_by": "Hamed_hamzeh",
    "build_timestamp_utc": ts,
}

# Write to file
meta_path = Path("bundle/metadata.json")
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metadata, indent=2))

{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "model_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "embedding_dim": 384,
  "max_seq_len": 256,
  "framework_version": "2.4.1+cpu",
  "transformers_version": "4.44.2",
  "built_by": "Hamed_hamzeh",
  "build_timestamp_utc": "2026-06-23T11:16:13Z"
}


# Step 3: Write MANIFEST.json

Compute SHA-256 hash for every file under `bundle/` (except MANIFEST.json itself)
and write the manifest to `bundle/MANIFEST.json`.

The format must be:
```json
{
  "format_version": 1,
  "files": {
    "relative/path/to/file": "sha256hexdigest",
    ...
  }
}
```

Pro tip: you can peek at `scripts/gen_manifest.py` for reference.

In [2]:
# Step 3: Clean unwanted files and generate final MANIFEST.json

import shutil

def sha256(filepath: Path) -> str:
    """Compute SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with filepath.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

root = Path("bundle")
model_dir = root / "model"

# Remove unwanted generated/cache files before manifest
shutil.rmtree(model_dir / ".cache", ignore_errors=True)

for unwanted in [
    model_dir / ".gitkeep",
    model_dir / ".commit",
]:
    if unwanted.exists():
        unwanted.unlink()

for pycache in root.rglob("__pycache__"):
    shutil.rmtree(pycache, ignore_errors=True)

for pyc in root.rglob("*.pyc"):
    pyc.unlink()

# Generate manifest only for real shipped files
files = {}

for p in sorted(root.rglob("*")):
    if not p.is_file():
        continue

    rel = p.relative_to(root).as_posix()

    if rel == "MANIFEST.json":
        continue
    if rel.startswith("model/.cache/"):
        continue
    if rel == "model/.gitkeep":
        continue
    if rel == "model/.commit":
        continue
    if "__pycache__/" in rel:
        continue
    if rel.endswith(".pyc"):
        continue

    files[rel] = sha256(p)

manifest = {
    "format_version": 1,
    "files": files,
}

manifest_path = root / "MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"MANIFEST.json written with {len(manifest['files'])} files")
for rel, h in sorted(manifest["files"].items()):
    print(f"  {h[:16]}...  {rel}")

MANIFEST.json written with 9 files
  fc0eb1a1463e76b9...  metadata.json
  953f9c0d463486b1...  model/config.json
  53aa51172d142c89...  model/model.safetensors
  303df45a03609e4e...  model/special_tokens_map.json
  be50c3628f2bf5bb...  model/tokenizer.json
  acb92769e8195aab...  model/tokenizer_config.json
  07eced375cec144d...  model/vocab.txt
  fa0468a834ce1857...  predict.py
  50cb137576a0093b...  requirements.txt


# Step 4: Write predict.py

Write `bundle/predict.py` with exactly 4 functions:

| Function | Signature | Returns |
|----------|-----------|---------|
| `load_bundle()` | no args | `(model, tokenizer)` tuple |
| `embed(texts)` | `List[str]` | `np.ndarray` shape `(N, 384)` float32 |
| `similarity(a, b)` | two `np.ndarray` | `float` (cosine similarity) |
| `info()` | no args | `dict` with metadata |

The **7-step pipeline** inside `embed()`:
1. Tokenize: `tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")`
2. Move tensors to device (cpu or cuda)
3. Forward pass under `torch.no_grad()` → `last_hidden_state`
4. Mean-pool: `sum(H * mask) / sum(mask).clamp(min=1e-9)`
5. L2 normalize: `F.normalize(pooled, p=2, dim=1)`
6. Detach, move to CPU, convert to `np.float32`
7. Return ndarray

**Important rules:**
- DO NOT import `sentence-transformers`. Use raw `transformers` only.
- Set `torch.manual_seed(0)` for determinism.
- Call `model.eval()` before inference.
- A template already exists at `bundle/predict.py` — open it and fill in the TODOs.

In [5]:
# test
import sys
sys.path.insert(0, "bundle")

from predict import load_bundle, embed, similarity, info

model, tokenizer = load_bundle()

print("Model loaded:", type(model).__name__)
print("Tokenizer loaded:", type(tokenizer).__name__)
print(info())

emb = embed(["hello world", "hello world"])
print("Embedding shape:", emb.shape)
print("Embedding dtype:", emb.dtype)
print("Similarity:", similarity(emb[0], emb[1]))

g:\work\MLops\mlops-bootcamp\hw22_deployment\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Model loaded: BertModel
Tokenizer loaded: BertTokenizerFast
{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41', 'embedding_dim': 384, 'max_seq_len': 256, 'framework_version': '2.4.1+cpu', 'transformers_version': '4.44.2', 'built_by': 'Hamed_hamzeh', 'build_timestamp_utc': '2026-06-23T11:16:13Z', 'device': 'cpu'}
Embedding shape: (2, 384)
Embedding dtype: float32
Similarity: 1.0000001192092896


# Step 5: Run tests

Run all 4 test files. All tests must pass with green dots.

Tests check:
- **test_parity.py** (7 tests): Correct embedding shape, L2 normalization, similarity
- **test_tokenization.py** (5 tests): Tokenizer behavior, special tokens, round-trip
- **test_determinism.py** (1 test): Same input → same output every time
- **test_adversarial.py** (10 tests): Edge cases (unicode, numerics, long text, empty strings)

In [6]:
# Run all tests
import os
os.environ["PYTHONPATH"] = "bundle"

!python -m pytest tests/ -v --tb=short

============================= test session starts =============================
platform win32 -- Python 3.11.7, pytest-8.3.2, pluggy-1.6.0 -- g:\work\MLops\mlops-bootcamp\hw22_deployment\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: g:\work\MLops\mlops-bootcamp\hw22_deployment\HW3_A
plugins: anyio-4.14.0
collecting ... collected 23 items

tests/test_adversarial.py::test_missing_bundle_dir PASSED                [  4%]
tests/test_adversarial.py::test_very_long_text PASSED                    [  8%]
tests/test_adversarial.py::test_unicode_text PASSED                      [ 13%]
tests/test_adversarial.py::test_numeric_text PASSED                      [ 17%]
tests/test_adversarial.py::test_single_token_text PASSED                 [ 21%]
tests/test_adversarial.py::test_duplicate_texts PASSED                   [ 26%]
tests/test_adversarial.py::test_batch_of_one PASSED                      [ 30%]
tests/test_adversarial.py::test_large_batch PASSED                       [ 34%]
tests/

# Step 6: Register in MLflow

Log the bundle to MLflow using `mlflow.sentence_transformers.log_model()`.

Before running this cell, make sure:
1. You have `source .env` (or set the env vars manually)
2. All tests pass
3. `bundle/model/` contains the 6 model files

In [4]:
# Register the bundle in MLflow
from dotenv import load_dotenv
from pathlib import Path
import os

import mlflow
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
BUNDLE_MODEL_DIR = Path("bundle/model")
BUNDLE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

env_path = Path(".env")

if env_path.exists():
    load_dotenv(env_path)
    print(".env loaded")
else:
    print(".env file not found in current directory:", Path.cwd())

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder"))

# load from local bundle (NOT used in predict.py, only for logging)
st_model = SentenceTransformer(str(BUNDLE_MODEL_DIR.resolve()))

with mlflow.start_run(run_name="hw3a-bundle") as run:
    mlflow.log_param("embedding_dim", 384)
    mlflow.log_param("max_seq_len", 256)
    mlflow.log_param("model_id", MODEL_ID)
    mlflow.set_tag("stage", "candidate")

    model_info = mlflow.sentence_transformers.log_model(
        model=st_model,
        artifact_path="bundle",
        task="llm/v1/embeddings",
    )

    print("Model URI:", model_info.model_uri)
    print("Run ID:", run.info.run_id)

.env loaded


No sentence-transformers model found with name G:\work\MLops\mlops-bootcamp\hw22_deployment\HW3_A\bundle\model. Creating a new one with mean pooling.
g:\work\MLops\mlops-bootcamp\hw22_deployment\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Model URI: runs:/cdd350808c574ca680ab1b30b5c56b51/bundle
Run ID: cdd350808c574ca680ab1b30b5c56b51


2026/06/24 18:57:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run hw3a-bundle at: http://185.50.38.163:33014/#/experiments/62/runs/cdd350808c574ca680ab1b30b5c56b51.
2026/06/24 18:57:15 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/62.


# Step 7: Upload to MinIO

Upload your `bundle/` directory to the shared MinIO bucket.

Run the provided upload script:
```bash
source .env && bash scripts/01_upload_to_minio.sh
```

Or from this notebook:

In [ ]:
# Upload bundle to MinIO using the provided script
env_path = Path(".env")

if env_path.exists():
    load_dotenv(env_path)
    print(".env loaded")
else:
    print(".env file not found in current directory:", Path.cwd())
    
!bash scripts/01_upload_to_minio.sh

# Take a screenshot of the upload confirmation for EVIDENCE/minio_upload.png

## Submission Checklist

Before submitting, verify:

- [ ] `encoder_bundle.ipynb` — all cells executed with visible outputs
- [ ] `bundle/predict.py` — 4 functions implemented
- [ ] `bundle/metadata.json` — all fields filled (no TODO placeholders)
- [ ] `bundle/MANIFEST.json` — real SHA-256 hashes for every file
- [ ] `bundle/requirements.txt` — pinned dependencies listed
- [ ] `bundle/model/` — 6 model files present
- [ ] `EVIDENCE/pytest_pass.png` — all tests green
- [ ] `EVIDENCE/mlflow_registered.png` — MLflow UI showing the model
- [ ] `EVIDENCE/minio_upload.png` — MinIO upload confirmation

Good luck!